## Data Cleaning & Transformation.

#### Clean and Transform Accounts Data

In [0]:
%python

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read from Bronze
accounts_bronze = spark.table("banking_catalog.bronze.accounts")

# Data Cleaning Operations
accounts_silver = accounts_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("accounts.csv"))

# Remove duplicates
accounts_silver = accounts_silver.dropDuplicates(["account_id"])

# Write to Silver
accounts_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@stbankinglake005.dfs.core.windows.net/accounts")

print(f"✅ Accounts Silver: {accounts_silver.count()} records")


✅ Accounts Silver: 500 records


#### Clean and Transform Customers Data

In [0]:
%python
# Read from Bronze
customers_bronze = spark.table("banking_catalog.bronze.customers")

# Data Cleaning
customers_silver = customers_bronze \
    .withColumn("name", trim(col("name"))) \
    .withColumn("gender", when(col("gender").isin(['M', 'F']), col("gender")).otherwise('U')) \
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", "")) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", upper(trim(col("state")))) \
    .withColumn("kyc_status", when(col("kyc_status").isin(['Y', 'N']), col("kyc_status")).otherwise('N')) \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("customers.csv"))

# Handle NULLs
customers_silver = customers_silver.fillna({
    "phone": "0000000000",
    "email": "unknown@email.com",
    "kyc_status": "N"
})

# Remove duplicates
customers_silver = customers_silver.dropDuplicates(["customer_id"])

# Write to Silver
customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@stbankinglake005.dfs.core.windows.net/customers")

print(f"✅ Customers Silver: {customers_silver.count()} records")


✅ Customers Silver: 500 records


#### Clean and Transform Transactions Data

In [0]:
%python
# Read from Bronze
transactions_bronze = spark.table("banking_catalog.bronze.transactions")

# Data Cleaning
transactions_silver = transactions_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Filter invalid transactions (negative amount or amount > 100000)
transactions_silver = transactions_silver \
    .filter(col("amount") >= 0) \
    .filter(col("amount") <= 100000) \
    .dropDuplicates(["transaction_id"])

# Write to Silver
transactions_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@stbankinglake005.dfs.core.windows.net/transactions")

print(f"✅ Transactions Silver: {transactions_silver.count()} records")


✅ Transactions Silver: 500 records


#### Clean and Transform Loans Data

In [0]:
%python
# Read from Bronze
loans_bronze = spark.table("banking_catalog.bronze.loans")

# Data Cleaning
loans_silver = loans_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'SAVINGS', 'CURRENT']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
loans_silver = loans_silver.dropDuplicates(["loan_id"])

# Write to Silver
loans_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@stbankinglake005.dfs.core.windows.net/loans")

print(f"✅ Loans Silver: {loans_silver.count()} records")


✅ Loans Silver: 500 records


#### Clean and Transform Fraud Alerts Data

In [0]:
%python
# Read from Bronze
fraud_bronze = spark.table("banking_catalog.bronze.fraud_alerts")

# Data Cleaning
fraud_silver = fraud_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("is_high_risk", when(col("amount") > 40000, lit("Y")).otherwise(lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
fraud_silver = fraud_silver.dropDuplicates(["alert_id"])

# Write to Silver
fraud_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@stbankinglake005.dfs.core.windows.net/fraud_alerts")

print(f"✅ Fraud Alerts Silver: {fraud_silver.count()} records")


✅ Fraud Alerts Silver: 500 records


#### Create Silver Tables in Unity Catalog referencing ADLS Silver layered data

In [0]:
%sql
-- Create Silver tables
CREATE TABLE banking_catalog.silver.accounts
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/accounts/';

CREATE TABLE banking_catalog.silver.customers
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/customers/';

CREATE TABLE banking_catalog.silver.transactions
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/transactions/';

CREATE TABLE banking_catalog.silver.loans
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/loans/';

CREATE TABLE banking_catalog.silver.fraud_alerts
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/fraud_alerts/';

CREATE TABLE banking_catalog.silver.atm_transactions
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/atm_transactions/';

CREATE TABLE banking_catalog.silver.branches
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/branches/';

CREATE TABLE banking_catalog.silver.credit_cards
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/credit_cards/';

CREATE TABLE banking_catalog.silver.kyc_documents
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/kyc_documents/';

CREATE TABLE banking_catalog.silver.employees
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/employees/';



In [0]:
%sql
CREATE TABLE IF NOT EXISTS banking_catalog.silver.dilip
USING DELTA
LOCATION 'abfss://silver@stbankinglake005.dfs.core.windows.net/dilip/';  -- Silver path